In [3]:
"""
Argentina EDA Analysis - World Values Survey
Exploratory Data Analysis of Argentina across 259 survey questions
comparing against global benchmarks from 66 countries
"""

import pandas as pd
import json
import numpy as np
from typing import Dict, Tuple, List

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================

def load_data(csv_path: str, json_path: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Load CSV data and JSON mapping file.
    
    Args:
        csv_path: Path to WVS data CSV file
        json_path: Path to question mapping JSON file
    
    Returns:
        Tuple of (DataFrame, mapping dictionary)
    """
    df = pd.read_csv(csv_path)
    
    with open(json_path, 'r') as f:
        mapping = json.load(f)
    
    return df, mapping


# ============================================================================
# 2. DATA EXPLORATION
# ============================================================================

def explore_data(df: pd.DataFrame) -> None:
    """Print basic information about the dataset."""
    print("=" * 80)
    print("DATASET OVERVIEW")
    print("=" * 80)
    print(f"\nDataset shape: {df.shape}")
    print(f"Number of countries: {df['B_COUNTRY_ALPHA'].nunique()}")
    print(f"Number of metrics: {len([col for col in df.columns if col.startswith('Q')])}")
    print(f"\nCountries in dataset:")
    print(df['B_COUNTRY_ALPHA'].unique())
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types summary:")
    print(df.dtypes.value_counts())


# ============================================================================
# 3. ARGENTINA-SPECIFIC ANALYSIS
# ============================================================================

def get_argentina_data(df: pd.DataFrame) -> pd.Series:
    """Extract Argentina's row from the dataset."""
    argentina = df[df['B_COUNTRY_ALPHA'] == 'Argentina'].iloc[0]
    return argentina


def get_valid_q_columns(df: pd.DataFrame) -> List[str]:
    """
    Extract valid Q columns (exclude metadata, regional indicators, etc.).
    
    Returns:
        List of column names starting with Q that aren't metadata
    """
    q_cols = [col for col in df.columns if col.startswith('Q')]
    
    # Filter out metadata and indicator columns
    exclude_patterns = ['_3', '_ABREV', '_LOCAL', 'CS9', 'R']
    valid_cols = [
        col for col in q_cols 
        if not any(pattern in col for pattern in exclude_patterns)
    ]
    
    return valid_cols


def calculate_argentina_metrics(
    df: pd.DataFrame, 
    argentina: pd.Series, 
    q_cols: List[str],
    mapping: Dict
) -> Dict:
    """
    Calculate comparative metrics for Argentina across all questions.
    
    Args:
        df: Full dataset
        argentina: Argentina's data row
        q_cols: List of valid Q columns
        mapping: Question mapping dictionary
    
    Returns:
        Dictionary with metrics for each question
    """
    argentina_metrics = {}
    
    for col in q_cols:
        if col not in argentina.index:
            continue
        
        arg_value = argentina[col]
        
        # Skip if Argentina's value is NaN
        if pd.isna(arg_value):
            continue
        
        # Calculate global statistics
        global_values = df[col].dropna()
        global_mean = global_values.mean()
        global_std = global_values.std()
        
        # Calculate percentile (% of countries with lower value)
        countries_unique = df.groupby('B_COUNTRY_ALPHA')[col].first()
        percentile = (countries_unique < arg_value).sum() / len(countries_unique) * 100
        
        argentina_metrics[col] = {
            'value': arg_value,
            'global_mean': global_mean,
            'global_std': global_std,
            'global_min': global_values.min(),
            'global_max': global_values.max(),
            'percentile': percentile,
            'z_score': (arg_value - global_mean) / global_std if global_std > 0 else 0,
            'description': mapping.get(col, 'N/A'),
            'distance_from_mean': abs(arg_value - global_mean)
        }
    
    return argentina_metrics


# ============================================================================
# 4. ANALYSIS AND RANKING
# ============================================================================

def get_top_strengths(
    metrics: Dict, 
    top_n: int = 10
) -> List[Tuple[str, Dict]]:
    """
    Get Argentina's top strengths (highest percentile rankings).
    
    Args:
        metrics: Metrics dictionary from calculate_argentina_metrics
        top_n: Number of top strengths to return
    
    Returns:
        List of (column_name, metric_data) tuples sorted by percentile
    """
    sorted_metrics = sorted(
        metrics.items(), 
        key=lambda x: x[1]['percentile'], 
        reverse=True
    )
    return sorted_metrics[:top_n]


def get_weaknesses(
    metrics: Dict, 
    bottom_n: int = 10
) -> List[Tuple[str, Dict]]:
    """
    Get Argentina's weaknesses (lowest percentile rankings).
    
    Args:
        metrics: Metrics dictionary from calculate_argentina_metrics
        bottom_n: Number of bottom weaknesses to return
    
    Returns:
        List of (column_name, metric_data) tuples sorted by percentile (ascending)
    """
    sorted_metrics = sorted(
        metrics.items(), 
        key=lambda x: x[1]['percentile']
    )
    return sorted_metrics[:bottom_n]


def get_midrange_metrics(
    metrics: Dict, 
    middle_n: int = 10
) -> List[Tuple[str, Dict]]:
    """
    Get metrics closest to global mean (most average performance).
    
    Args:
        metrics: Metrics dictionary
        middle_n: Number of metrics to return
    
    Returns:
        List of (column_name, metric_data) tuples sorted by distance from mean
    """
    sorted_metrics = sorted(
        metrics.items(), 
        key=lambda x: x[1]['distance_from_mean']
    )
    return sorted_metrics[:middle_n]


# ============================================================================
# 5. VISUALIZATION AND REPORTING
# ============================================================================

def print_strengths_report(strengths: List[Tuple[str, Dict]]) -> None:
    """Print formatted report of Argentina's strengths."""
    print("\n" + "=" * 100)
    print("TOP STRENGTHS (Highest Percentile Rankings)")
    print("=" * 100)
    
    for i, (col, data) in enumerate(strengths, 1):
        print(f"\n{i}. {col}: {data['value']:.4f} (Percentile: {data['percentile']:.1f}%)")
        print(f"   Description: {data['description']}")
        print(f"   Global Mean: {data['global_mean']:.4f} | Std: {data['global_std']:.4f}")
        print(f"   Z-Score: {data['z_score']:.4f}")


def print_weaknesses_report(weaknesses: List[Tuple[str, Dict]]) -> None:
    """Print formatted report of Argentina's weaknesses."""
    print("\n" + "=" * 100)
    print("TOP WEAKNESSES (Lowest Percentile Rankings)")
    print("=" * 100)
    
    for i, (col, data) in enumerate(weaknesses, 1):
        print(f"\n{i}. {col}: {data['value']:.4f} (Percentile: {data['percentile']:.1f}%)")
        print(f"   Description: {data['description']}")
        print(f"   Global Mean: {data['global_mean']:.4f} | Std: {data['global_std']:.4f}")
        print(f"   Z-Score: {data['z_score']:.4f}")


def print_midrange_report(midrange: List[Tuple[str, Dict]]) -> None:
    """Print formatted report of average metrics."""
    print("\n" + "=" * 100)
    print("MIDRANGE METRICS (Closest to Global Mean)")
    print("=" * 100)
    
    for i, (col, data) in enumerate(midrange, 1):
        print(f"\n{i}. {col}: {data['value']:.4f} (Percentile: {data['percentile']:.1f}%)")
        print(f"   Description: {data['description']}")
        print(f"   Distance from Mean: {data['distance_from_mean']:.4f}")
        print(f"   Global Mean: {data['global_mean']:.4f}")


def create_summary_table(
    strengths: List[Tuple[str, Dict]], 
    weaknesses: List[Tuple[str, Dict]]
) -> pd.DataFrame:
    """
    Create a summary DataFrame of top strengths and weaknesses.
    
    Args:
        strengths: List of strength tuples
        weaknesses: List of weakness tuples
    
    Returns:
        DataFrame with summary statistics
    """
    summary_data = []
    
    # Add strengths
    for col, data in strengths[:5]:
        summary_data.append({
            'Category': 'Strength',
            'Metric': col,
            'Argentina Value': f"{data['value']:.4f}",
            'Global Mean': f"{data['global_mean']:.4f}",
            'Percentile': f"{data['percentile']:.1f}%",
            'Z-Score': f"{data['z_score']:.4f}"
        })
    
    # Add weaknesses
    for col, data in weaknesses[:5]:
        summary_data.append({
            'Category': 'Weakness',
            'Metric': col,
            'Argentina Value': f"{data['value']:.4f}",
            'Global Mean': f"{data['global_mean']:.4f}",
            'Percentile': f"{data['percentile']:.1f}%",
            'Z-Score': f"{data['z_score']:.4f}"
        })
    
    return pd.DataFrame(summary_data)


# ============================================================================
# 6. STATISTICAL SUMMARY
# ============================================================================

def print_statistical_summary(
    df: pd.DataFrame, 
    argentina: pd.Series,
    valid_q_cols: List[str]
) -> None:
    """Print overall statistical summary for Argentina."""
    print("\n" + "=" * 100)
    print("STATISTICAL SUMMARY - ARGENTINA vs GLOBAL")
    print("=" * 100)
    
    # Calculate Argentina's mean across all metrics
    argentina_values = [argentina[col] for col in valid_q_cols if not pd.isna(argentina[col])]
    argentina_mean = np.mean(argentina_values)
    argentina_std = np.std(argentina_values)
    
    # Calculate global means
    global_means = []
    for col in valid_q_cols:
        global_mean = df[col].mean()
        global_means.append(global_mean)
    
    global_mean = np.mean(global_means)
    global_std = np.mean([df[col].std() for col in valid_q_cols])
    
    print(f"\nArgentina across {len(argentina_values)} metrics:")
    print(f"  Mean: {argentina_mean:.4f}")
    print(f"  Std Dev: {argentina_std:.4f}")
    print(f"  Min Value: {min(argentina_values):.4f}")
    print(f"  Max Value: {max(argentina_values):.4f}")
    
    print(f"\nGlobal (all {df['B_COUNTRY_ALPHA'].nunique()} countries):")
    print(f"  Mean of Country Means: {global_mean:.4f}")
    print(f"  Mean of Country Std Devs: {global_std:.4f}")
    
    print(f"\nDifference (Argentina - Global):")
    print(f"  Mean Difference: {argentina_mean - global_mean:.4f}")


# ============================================================================
# 7. MAIN EXECUTION
# ============================================================================

def main(csv_path: str = 'C:\\Users\\amaca253\\Documents\\datomatisation\\datomatisation-dev\\data\\demo_data\\wvs\\data-wvs.csv', json_path: str = 'C:\\Users\\amaca253\\Documents\\datomatisation\\datomatisation-dev\\data\\demo_data\\wvs\\map-wvs.json'):
    """
    Main execution function for Argentina EDA.
    
    Args:
        csv_path: Path to WVS CSV data file
        json_path: Path to WVS question mapping JSON file
    """
    # Load data
    print("\nLoading data...")
    df, mapping = load_data(csv_path, json_path)
    
    # Explore dataset
    explore_data(df)
    
    # Get Argentina's data
    print("\n" + "=" * 80)
    print("ARGENTINA ANALYSIS")
    print("=" * 80)
    argentina = get_argentina_data(df)
    
    # Get valid Q columns
    valid_q_cols = get_valid_q_columns(df)
    print(f"\nAnalyzing {len(valid_q_cols)} valid metrics...")
    
    # Calculate metrics
    metrics = calculate_argentina_metrics(df, argentina, valid_q_cols, mapping)
    print(f"Successfully calculated metrics for {len(metrics)} questions")
    
    # Get analysis results
    strengths = get_top_strengths(metrics, top_n=10)
    weaknesses = get_weaknesses(metrics, bottom_n=10)
    midrange = get_midrange_metrics(metrics, middle_n=10)
    
    # Print reports
    print_strengths_report(strengths)
    print_weaknesses_report(weaknesses)
    print_midrange_report(midrange)
    print_statistical_summary(df, argentina, valid_q_cols)
    
    # Create summary table
    print("\n" + "=" * 100)
    print("SUMMARY TABLE")
    print("=" * 100)
    summary_df = create_summary_table(strengths, weaknesses)
    print(summary_df.to_string(index=False))
    
    return {
        'data': df,
        'mapping': mapping,
        'argentina': argentina,
        'metrics': metrics,
        'strengths': strengths,
        'weaknesses': weaknesses,
        'midrange': midrange,
        'summary_table': summary_df
    }


# ============================================================================
# 8. FILTER AND CATEGORY ANALYSIS (Optional)
# ============================================================================

def analyze_by_category(
    metrics: Dict, 
    category_keywords: Dict[str, List[str]]
) -> Dict:
    """
    Analyze metrics by category based on question keywords.
    
    Args:
        metrics: Metrics dictionary
        category_keywords: Dict mapping category names to Q column keywords
    
    Returns:
        Dictionary with metrics grouped by category
    """
    categorized = {cat: {} for cat in category_keywords.keys()}
    
    for col, data in metrics.items():
        for category, keywords in category_keywords.items():
            if any(keyword.lower() in data['description'].lower() for keyword in keywords):
                categorized[category][col] = data
                break
    
    return categorized


def print_category_summary(categorized: Dict) -> None:
    """Print summary of metrics by category."""
    print("\n" + "=" * 100)
    print("ANALYSIS BY CATEGORY")
    print("=" * 100)
    
    for category, metrics_dict in categorized.items():
        if not metrics_dict:
            continue
        
        percentiles = [m['percentile'] for m in metrics_dict.values()]
        
        print(f"\n{category}:")
        print(f"  Count: {len(metrics_dict)}")
        print(f"  Mean Percentile: {np.mean(percentiles):.1f}%")
        print(f"  Min Percentile: {min(percentiles):.1f}%")
        print(f"  Max Percentile: {max(percentiles):.1f}%")


# ============================================================================
# RUN ANALYSIS
# ============================================================================

if __name__ == "__main__":
    results = main(csv_path='C:\\Users\\amaca253\\Documents\\datomatisation\\datomatisation-dev\\data\\demo_data\\wvs\\data-wvs.csv', json_path='C:\\Users\\amaca253\\Documents\\datomatisation\\datomatisation-dev\\data\\demo_data\\wvs\\map-wvs.json')
    
    # Optional: Category analysis
    category_keywords = {
        'Tolerance & Neighbors': ['neighbors', 'race', 'religion', 'immigrants'],
        'Trust & Social': ['trust', 'family', 'friends'],
        'Values & Children': ['children', 'qualities'],
        'Economic & Safety': ['income', 'food', 'shelter', 'crime', 'unsafe'],
        'Political & Democracy': ['democracy', 'election', 'government', 'political'],
        'Well-being': ['happy', 'health', 'satisfaction', 'choice']
    }
    
    categorized = analyze_by_category(results['metrics'], category_keywords)
    print_category_summary(categorized)


Loading data...
DATASET OVERVIEW

Dataset shape: (66, 321)
Number of countries: 66
Number of metrics: 320

Countries in dataset:
['Andorra' 'Argentina' 'Armenia' 'Australia' 'Bangladesh'
 'Plurinational State of Bolivia' 'Brazil' 'Canada' 'Chile' 'China'
 'Colombia' 'Cyprus' 'Czechia' 'Germany' 'Ecuador' 'Egypt' 'Ethiopia'
 'United Kingdom' 'Greece' 'Guatemala' 'China, Hong Kong SAR' 'Indonesia'
 'India' 'Islamic Republic of Iran' 'Iraq' 'Jordan' 'Japan' 'Kazakhstan'
 'Kenya' 'Kyrgyzstan' 'Republic of Korea' 'Lebanon' 'Libya'
 'China, Macao SAR' 'Morocco' 'Maldives' 'Mexico' 'Myanmar' 'Mongolia'
 'Malaysia' 'Nigeria' 'Nicaragua' 'Northern Ireland' 'Netherlands'
 'New Zealand' 'Pakistan' 'Peru' 'Philippines' 'Puerto Rico' 'Romania'
 'Russia' 'Singapore' 'Serbia' 'Slovakia' 'Thailand' 'Tajikistan'
 'Tunisia' 'Turkey' 'Taiwan' 'Ukraine' 'Uruguay'
 'United States of America' 'Uzbekistan'
 'Bolivarian Republic of Venezuela' 'Vietnam' 'Zimbabwe']

First few rows:
  B_COUNTRY_ALPHA  Q_MODE  